In [ ]:
# =====================================================
# E-COMMERCE RFM ANALYSIS (FINAL ROBUST VERSION)
# =====================================================

import pandas as pd

print("🚀 Starting Pipeline...")

# =====================================================
# 1. LOAD DATA
# =====================================================
df_orders = pd.read_csv("data_raw/olist_orders_dataset.csv")
df_customers = pd.read_csv("data_raw/olist_customers_dataset.csv")
df_items = pd.read_csv("data_raw/olist_order_items_dataset.csv")

# Clean column names
df_orders.columns = df_orders.columns.str.strip()
df_customers.columns = df_customers.columns.str.strip()
df_items.columns = df_items.columns.str.strip()

print("✅ Data Loaded")

# =====================================================
# 2. DATE CONVERSION (FINAL FIX)
# =====================================================

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

print("✅ Dates Converted Cleanly")
# =====================================================
# 3. FEATURE ENGINEERING
# =====================================================
df_orders['delivery_time_days'] = (
    df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']
).dt.days

df_orders['processing_time_days'] = (
    df_orders['order_approved_at'] - df_orders['order_purchase_timestamp']
).dt.days

# Keep only valid delivered orders
df_delivered = df_orders[
    (df_orders['order_status'] == 'delivered') &
    (df_orders['delivery_time_days'] >= 0) &
    (df_orders['processing_time_days'] >= 0)
].copy()

print("✅ Valid Orders Filtered")

# =====================================================
# 4. MERGE CUSTOMER DATA
# =====================================================
df_merged = df_delivered.merge(
    df_customers,
    on='customer_id',
    how='left'
)

print("✅ Customer Merge Done")

# =====================================================
# 5. MONETARY (ORDER VALUE)
# =====================================================
if 'price' not in df_items.columns:
    raise Exception(f"❌ 'price' column missing. Columns found: {df_items.columns}")

df_order_value = df_items.groupby('order_id')['price'].sum().reset_index()
df_order_value.rename(columns={'price': 'order_value'}, inplace=True)

df_full = df_merged.merge(df_order_value, on='order_id', how='left')
df_full['order_value'] = df_full['order_value'].fillna(0)

print("✅ Monetary Calculated")

# =====================================================
# 6. CUSTOMER LEVEL DATA (RFM BASE)
# =====================================================
df_customer = df_full.groupby('customer_unique_id').agg({
    'order_id': 'count',
    'order_purchase_timestamp': 'max',
    'order_value': 'sum'
}).reset_index()

df_customer.rename(columns={
    'order_id': 'frequency',
    'order_purchase_timestamp': 'last_purchase_date',
    'order_value': 'monetary'
}, inplace=True)

print("✅ Customer Aggregation Done")

# =====================================================
# 7. RECENCY
# =====================================================
today = df_full['order_purchase_timestamp'].max()

df_customer['recency'] = (
    today - df_customer['last_purchase_date']
).dt.days

print("✅ Recency Done")

# =====================================================
# 8. RFM SCORING (ROBUST FIX)
# =====================================================
# Ranking to avoid duplicate issues
df_customer['R_rank'] = df_customer['recency'].rank(method='first')
df_customer['F_rank'] = df_customer['frequency'].rank(method='first')
df_customer['M_rank'] = df_customer['monetary'].rank(method='first')

df_customer['R_score'] = pd.qcut(df_customer['R_rank'], 5, labels=[5,4,3,2,1])
df_customer['F_score'] = pd.qcut(df_customer['F_rank'], 5, labels=[1,2,3,4,5])
df_customer['M_score'] = pd.qcut(df_customer['M_rank'], 5, labels=[1,2,3,4,5])

df_customer['RFM_score'] = (
    df_customer['R_score'].astype(str) +
    df_customer['F_score'].astype(str) +
    df_customer['M_score'].astype(str)
)

print("✅ RFM Scoring Done")

# =====================================================
# 9. SEGMENTATION
# =====================================================
def segment_customer(row):
    if row['RFM_score'] in ['555', '554', '544', '545']:
        return 'Champions'
    elif row['RFM_score'] in ['543', '444', '435']:
        return 'Loyal Customers'
    elif row['RFM_score'] in ['355', '354', '345']:
        return 'Potential Loyalist'
    elif row['RFM_score'] in ['255', '254', '245']:
        return 'New Customers'
    elif row['RFM_score'] in ['155', '154']:
        return 'At Risk'
    else:
        return 'Others'

df_customer['segment'] = df_customer.apply(segment_customer, axis=1)

print("✅ Segmentation Done")

# =====================================================
# 10. FINAL OUTPUT
# =====================================================
print("\n📊 FINAL DATA SHAPE:", df_customer.shape)

print("\n📌 SAMPLE DATA:")
print(df_customer.head())

print("\n📈 SEGMENT DISTRIBUTION:")
print(df_customer['segment'].value_counts())

import os

os.makedirs("data_cleaned", exist_ok=True)

df_customer.to_csv("data_cleaned/rfm_customer_data.csv", index=False)

print("✅ RFM data exported successfully")

🚀 Starting Pipeline...
✅ Data Loaded
✅ Dates Converted Cleanly
✅ Valid Orders Filtered
✅ Customer Merge Done
✅ Monetary Calculated
✅ Customer Aggregation Done
✅ Recency Done
✅ RFM Scoring Done
✅ Segmentation Done

📊 FINAL DATA SHAPE: (93337, 13)

📌 SAMPLE DATA:
                 customer_unique_id  frequency  last_purchase_date  monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2          1 2018-05-10 10:56:27    129.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f          1 2018-05-07 11:11:27     18.90   
2  0000f46a3911fa3c0805444483337064          1 2017-03-10 21:05:03     69.00   
3  0000f6ccb0745a6a4b88665a16c9f078          1 2017-10-12 20:29:41     25.99   
4  0004aac84e0df4da2b147fca70cf8255          1 2017-11-14 19:45:42    180.00   

   recency   R_rank  F_rank   M_rank R_score F_score M_score RFM_score segment  
0      111  22399.0     1.0  62926.0       4       1       4       414  Others  
1      114  23391.0     2.0   3923.0       4       1       1       411  Others  
2      536  90